# Match-level timeseries across samples

This notebook plots matched peak intensities across samples at three granularity levels:

| Level        | Column                    | Description                                    |
| ------------ | ------------------------- | ---------------------------------------------- |
| **Isotope**  | `target_isotope_formula`  | One-to-one with peaks matched to isotopes      |
| **Ion**      | `target_ion_formula`      | Groups isotopes of the same ion                |
| **Compound** | `target_compound_formula` | Groups all ions belonging to the same compound |

Each section aggregates peak heights per sample at the chosen level and plots them over time.


### Load peaks

Each peak carries three match scores (0 – 1) at different aggregation levels:

| Score                  | Meaning                                                            |
| ---------------------- | ------------------------------------------------------------------ |
| `match_score_isotope`  | How well a single isotope's m/z and abundance match with the peak  |
| `match_score_ion`      | Weighted sum of isotope scores (by relative abundance) for the ion |
| `match_score_compound` | Max of ion scores across all ions of the compound                  |

After loading we define a threshold per level and apply it in each section below.


In [ ]:
import plotly.express as px
import plotly.io as pio

from mascope_sdk import MascopeClient


pio.templates.default = "plotly_dark"  # or "plotly_white"

mascope = MascopeClient(workspace="My Workspace")

# Load peaks across batches (adjust dataset/batches to your data)
peaks = mascope.load_peaks(dataset="My Dataset", batches="My Batch")

In [ ]:
# Keep only matched peaks
matched = peaks[peaks["target_compound_formula"].notna()].copy()

# Composite label: shows the compound + ionization path that produced the ion formula.
# Two different compounds can yield the same ion formula via different mechanisms;
# this column makes each path visible in plots.
matched["ion_formula_path"] = (
    matched["target_compound_formula"] + matched["ionization_mechanism"]
)

matched

In [ ]:
# Optional: narrow down to specific compounds, ions, or isotopes
# Examples::
# matched = matched[matched["target_compound_name"].isin(["Acetone", "Methanol"])]
# matched = matched[matched["target_ion_formula"].isin(["C3H7O+", "CH5O+"])]

# See what's available at each level:
for col in ["target_compound_formula", "target_ion_formula", "target_isotope_formula"]:
    vals = matched[col].unique()
    print(f"{col}: {len(vals)} unique")

matched

### Isotope-level

Each isotope maps one-to-one to a peak, so this is the finest granularity.


In [ ]:
# Match score threshold (0 – 1)
isotope_score_min = 0.9

isotope_filtered = matched[matched["match_score_isotope"] >= isotope_score_min]

isotope_ts = isotope_filtered.reset_index(drop=True).sort_values("datetime_utc")

fig = px.scatter(
    isotope_ts,
    x="datetime_utc",
    y="height",
    color="target_isotope_formula",
    hover_data=["sample_item_name"],
    title="Isotope-level intensities across samples",
)
# Truncate long names to avoid legend overflow
max_name_len = 30
for trace in fig.data:
    if len(trace.name) > max_name_len:
        trace.name = trace.name[:max_name_len] + "…"

fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig.show()

### Ion-level

Aggregate isotopes belonging to the same ion (sum of isotopes).

**Note:** Different compounds can produce the same ion formula (via different ionization mechanisms). A peak matching such an ion contributes to _both_ ions, so ion-level heights are not strictly additive. This is analytically correct - the signal is genuinely consistent with both ionization paths - but keep it in mind when comparing totals, and deduplicate where needed.


In [ ]:
# Match score threshold (0 – 1)
ion_score_min = 0.9

ion_filtered = matched[matched["match_score_ion"] >= ion_score_min]

ion_ts = (
    ion_filtered.groupby(
        ["datetime_utc", "sample_item_name", "ion_formula_path", "target_ion_formula"]
    )["height"]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula"],
    title="Ion-level intensities across samples",
)
fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig.show()

### Compound-level

Aggregate all peaks belonging to the same compound (sum of ions).

**Note:** Different compounds can produce the same ion formula (via different ionization mechanisms). A peak matching such an ion contributes to _both_ compounds, so compound-level heights are not strictly additive across compounds.


In [ ]:
# Match score threshold (0 – 1)
compound_score_min = 0.9

compound_filtered = matched[matched["match_score_compound"] >= compound_score_min]

compound_ts = (
    compound_filtered.groupby(
        [
            "datetime_utc",
            "sample_item_name",
            "target_compound_id",
            "target_compound_formula",
        ]
    )["height"]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    compound_ts,
    x="datetime_utc",
    y="height",
    color="target_compound_formula",
    hover_data=["sample_item_name"],
    title="Compound-level intensities across samples",
)
fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig.show()

### Time-averaged view

Resample any of the above timeseries to a coarser time bin. This example uses the ion-level data.


In [ ]:
import pandas as pd


# Resample to a coarser time bin (e.g. "1h", "10min")
time_aggregation = "1h"

ion_ts_avg = (
    ion_ts.groupby(
        [
            "ion_formula_path",
            pd.Grouper(key="datetime_utc", freq=time_aggregation),
        ]
    )["height"]
    .mean()
    .reset_index()
    .dropna()
)

fig = px.scatter(
    ion_ts_avg,
    x="datetime_utc",
    y="height",
    color="ion_formula_path",
    title=f"Ion intensities - {time_aggregation} average",
)
fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity (mean) [cps]")
fig.show()